Assignment : KNN & PCA

Question 1: What is K-Nearest Neighbors (KNN) and how does it work in both classification and regression problems?

Ans - K-Nearest Neighbors is a supervised, instance-based learning algorithm. It stores the labeled training observations and predicts for a new observation by finding the k training observations closest to it under a chosen distance measure. Because the method relies directly on distances, feature scales and the choice of k and distance metric can strongly affect its predictions.
In classification, each neighbor votes for its class. The predicted class is the class with the most votes; a distance-weighted variant gives closer neighbors more influence. In regression, the prediction is usually the arithmetic mean of the neighbors’ target values, or a distance-weighted mean. A small k can make predictions sensitive to noise, while a larger k smooths the decision boundary but may blur local structure. KNN has little training cost, but prediction can be expensive because distances to many training points must be computed and stored.

Question 2: What is the Curse of Dimensionality and how does it affect KNN performance?

Ans - The Curse of Dimensionality describes difficulties that arise as the number of feature dimensions grows. The volume of the feature space expands rapidly, so a fixed amount of data becomes sparse. To maintain comparable coverage, the number of observations needed can grow dramatically with dimension.
This harms KNN because “nearest” becomes less informative: distances tend to become more similar, and many observations may be far from the query in absolute terms. Irrelevant or noisy variables can dominate the distance calculation, while a finite training set provides few truly comparable neighbors. As a result, generalization can worsen and predictions may become unstable. Computation and memory also increase with the number of features and stored samples.
Mitigations include removing irrelevant features, scaling variables, collecting more representative observations, and reducing dimension with methods such as PCA. The reduction step must be fitted using training data only. Validation should compare the complete preprocessing-plus-model pipeline, since removing dimensions can discard useful signal as well as noise.


Question 3: What is Principal Component Analysis (PCA)? How is it different from feature selection?

Ans - Principal Component Analysis is an unsupervised linear transformation that represents correlated numeric variables using new orthogonal axes called principal components. The first component points in the direction of greatest variance in the data; each following component captures the greatest remaining variance subject to being orthogonal to earlier components. Data can be projected onto only the first few components to create a lower-dimensional representation.
Feature selection keeps a subset of the original variables, such as selected gene measurements, and discards the rest. PCA instead constructs new variables as weighted combinations of the original variables. Thus, PCA can compress correlated information efficiently, but its components may be less directly interpretable than original features. PCA uses no class labels when finding directions of variance, so high variance is not guaranteed to be the most predictive variance. Scaling is usually important when variables have different units or ranges.


Question 4: What are eigenvalues and eigenvectors in PCA, and why are they important?

Ans - After centering (and often standardizing) the data, PCA forms a covariance or correlation matrix. An eigenvector of this matrix gives a direction in feature space; its corresponding eigenvalue measures the variance along that direction. PCA orders eigenvectors by decreasing eigenvalue and uses them as principal-component directions.
The largest eigenvalue corresponds to the direction capturing the most variance. The explained-variance ratio for component i is its eigenvalue divided by the sum of all eigenvalues. These ratios quantify how much total variance each component retains and help guide a dimensionality choice. Eigenvectors provide the component loadings, showing how original variables contribute to each direction. Eigenvalues and eigenvectors therefore define both the projection and the information-retention calculation.


Question 5
How do KNN and PCA complement each other when applied in a single pipeline?

Ans - PCA can make KNN more effective when many input variables are redundant or noisy. It transforms the standardized training data into a smaller set of orthogonal coordinates; KNN then measures distances in that reduced representation. This can lower computational cost, reduce the influence of correlated variables, and sometimes improve generalization by removing low-variance noise.
A sound pipeline is: split the data; fit scaling on the training fold; fit PCA on the scaled training fold; transform training and validation data with those fitted steps; fit KNN on the transformed training data; and evaluate on held-out observations. Scaling, PCA, and KNN choices should be tuned together using cross-validation inside the training data. PCA can also hurt if discarded components carry class information, so performance must be compared with a scaled KNN model without PCA. In high-dimensional, small-sample settings, leakage prevention and appropriately repeated or nested validation are essential


In [1]:
#Dataset: Use the Wine Dataset from sklearn.datasets.load_wine().
#Question 6: Train a KNN Classifier on the Wine dataset with and without feature scaling. Compare model accuracy in both cases.

#KNN uses distances, so features with large numeric ranges can dominate an unscaled model. The code below uses the same stratified holdout split and k=5 in both conditions. StandardScaler is fitted on the training data only through the pipeline, preventing test-set information from influencing the transformation.

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "without scaling": make_pipeline(KNeighborsClassifier(n_neighbors=5)),
    "with scaling": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: {model.score(X_test, y_test):.4f}")


without scaling: 0.7778
with scaling: 0.9333


In [2]:
#Question 7: Train a PCA model on the Wine dataset and print the explained variance ratio of each principal component.

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

from sklearn.decomposition import PCA

pca = make_pipeline(StandardScaler(), PCA())
pca.fit(X_train, y_train)
ratios = pca.named_steps["pca"].explained_variance_ratio_
for i, ratio in enumerate(ratios, start=1):
    print(f"PC{i}: {ratio:.6f}")
print(f"Total: {ratios.sum():.6f}")


PC1: 0.357499
PC2: 0.192093
PC3: 0.108456
PC4: 0.074183
PC5: 0.069357
PC6: 0.052031
PC7: 0.043915
PC8: 0.025006
PC9: 0.022021
PC10: 0.019160
PC11: 0.016517
PC12: 0.012489
PC13: 0.007273
Total: 1.000000


In [3]:
#Question 8: Train a KNN Classifier on the PCA-transformed dataset (retain top 2 components). Compare the accuracy with the original dataset.

#This experiment standardizes the data, fits PCA with two components on the training portion, and compares k=5 KNN on the two-dimensional projection against k=5 KNN on all 13 standardized features. Both models use exactly the same train-test split.

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

from sklearn.decomposition import PCA

original_model = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=5)
)
pca_model = make_pipeline(
    StandardScaler(), PCA(n_components=2),
    KNeighborsClassifier(n_neighbors=5)
)
for name, model in [("original scaled features", original_model),
                    ("PCA with 2 components", pca_model)]:
    model.fit(X_train, y_train)
    print(f"{name}: {model.score(X_test, y_test):.4f}")


original scaled features: 0.9333
PCA with 2 components: 0.9333


In [4]:
#Question 9: Train a KNN Classifier with different distance metrics (euclidean, manhattan) on the scaled Wine dataset and compare the results.

#Euclidean distance measures straight-line distance, whereas Manhattan distance sums absolute coordinate differences. The following comparison keeps the split, scaler, and number of neighbors fixed so the distance metric is the only changed setting.

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

for metric in ("euclidean", "manhattan"):
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5, metric=metric)
    )
    model.fit(X_train, y_train)
    print(f"{metric}: {model.score(X_test, y_test):.4f}")


euclidean: 0.9333
manhattan: 0.9778


Question 10: You are working with a high-dimensional gene expression dataset to classify patients with different types of cancer. Due to the large number of features and a small number of samples, traditional models overfit.

Explain how you would:
● Use PCA to reduce dimensionality
● Decide how many components to keep
● Use KNN for classification post-dimensionality reduction
● Evaluate the model
● Justify this pipeline to your stakeholders as a robust solution for real-world biomedical data


Answer
Use a leakage-safe, nested validation design. Keep an untouched external cohort for final assessment if one is available. In each training fold, standardize expression measurements using that fold’s training samples, fit PCA on those samples only, and transform the validation samples with the fitted steps. With thousands of genes and far fewer samples, PCA can be computed efficiently with an SVD-based implementation; it reduces correlated measurements to orthogonal components. It is not supervised, so component selection must be judged by predictive performance, not variance alone.
Choose the retained dimensionality by comparing pre-specified variance thresholds (for example 80%, 90%, 95%, and 99%) inside the inner cross-validation loop. Tune KNN’s neighbor count and distance metric in that same inner loop. The outer loop estimates performance for the entire selection procedure and limits optimism from tuning. If class counts are too small for five folds, use fewer stratified folds and report that limitation; never split observations from the same patient across train and validation. Where samples are grouped by patient, site, or batch, use group-aware splitting.
Report balanced accuracy, macro F1, per-class sensitivity/recall, and a confusion matrix alongside ordinary accuracy, since class imbalance can make accuracy misleading. Summarize outer-fold scores with mean and dispersion or confidence intervals, inspect class-wise failures, and compare with a simple baseline and a scaled KNN without PCA. If possible, evaluate once on an independent cohort and assess calibration and subgroup behavior. Biomedical claims require external validation, data-quality checks, and expert review; PCA plus KNN is a defensible candidate pipeline, not proof of clinical utility.
The code below expects the gene-expression matrix and labels to be supplied as X_gene (rows are patient samples; columns are genes) and y_gene. It uses nested stratified cross-validation to evaluate the full pipeline. Because the assignment does not provide this dataset, its numerical output cannot be truthfully filled in here; running the code with the actual data prints the measured scores.


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                             accuracy_score)

# Supply X_gene and y_gene from the study dataset.
# X_gene: (number of patients, number of genes); y_gene: cancer labels.
X_gene = np.asarray(X_gene)
y_gene = np.asarray(y_gene)
class_counts = np.unique(y_gene, return_counts=True)[1]
if class_counts.min() < 5:
    raise ValueError("At least 5 patients per class are needed for this 5-fold example; reduce folds carefully if needed.")

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(svd_solver="full")),
    ("knn", KNeighborsClassifier()),
])
param_grid = {
    "pca__n_components": [0.80, 0.90, 0.95, 0.99],
    "knn__n_neighbors": [3, 5, 7],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=43)
results = []
for train_idx, test_idx in outer.split(X_gene, y_gene):
    search = GridSearchCV(pipe, param_grid, cv=inner,
                          scoring="balanced_accuracy", n_jobs=-1)
    search.fit(X_gene[train_idx], y_gene[train_idx])
    pred = search.predict(X_gene[test_idx])
    results.append((accuracy_score(y_gene[test_idx], pred),
                    balanced_accuracy_score(y_gene[test_idx], pred),
                    f1_score(y_gene[test_idx], pred, average="macro")))
results = np.asarray(results)
print("Outer CV mean accuracy, balanced accuracy, macro F1:",
      np.round(results.mean(axis=0), 3))
print("Outer CV standard deviations:", np.round(results.std(axis=0), 3))
